In [ ]:
ref_geojson = "Update2025/historisierte-administrative_grenzen_k4_2025-04-06_gemeinde_4326.geojson"
gmdnr = "GDENR"
gmdname = "GDENAME"
env = "dev"

import ManageAdminBoundaries as ab
ab.CheckMunicipalityBoundaries(ref_geojson, gmdnr, gmdname, env)

In [ ]:
ref_geojson = "Update2025/historisierte-administrative_grenzen_k4_2025-04-06_bezirk_4326.geojson"
bznr = "BEZNR"
bzname = "BEZNAME"
env = "dev"

import ManageAdminBoundaries as ab
ab.CheckDistrictBoundaries(ref_geojson, bznr, bzname, env)

In [1]:
ref_geojson = "Update2025/historisierte-administrative_grenzen_k4_2025-04-06_kanton_4326.geojson"
ktnr = "KTNR"
ktname = "KTKZ"
env = "dev"

import ManageAdminBoundaries as ab
ab.CheckCantonBoundaries(ref_geojson, ktnr, ktname, env)

Exporting all cantons from geocat : 99.0%Done
Checking cantons...Done


In [ ]:
ref_geojson = "Update2025/historisierte-administrative_grenzen_k4_2025-04-06_gemeinde_4326.geojson"
number = "GDENR"
name = "GDENAME"
type = "g"
env = "dev"
update_name = True

import ManageAdminBoundaries as ab
manage = ab.UpdateSubtemplatesExtent(ref_geojson, number, name, type, update_name, env)
manage.update_all_subtemplates(with_backup=False)

In [ ]:
ref_geojson = "Update2025/historisierte-administrative_grenzen_k4_2025-04-06_bezirk_4326.geojson"
number = "BEZNR"
name = "BEZNAME"
type = "b"
env = "dev"
update_name = True

import ManageAdminBoundaries as ab
manage = ab.UpdateSubtemplatesExtent(ref_geojson, number, name, type, update_name, env)
manage.update_all_subtemplates(with_backup=False)

In [2]:
ref_geojson = "Update2025/historisierte-administrative_grenzen_k4_2025-04-06_kanton_4326.geojson"
number = "KTNR"
name = "KTKZ"
env = "dev"
type = "k"
env = "dev"
update_name = True

import ManageAdminBoundaries as ab
manage = ab.UpdateSubtemplatesExtent(ref_geojson, number, name, type, update_name, env)
manage.update_all_subtemplates(with_backup=True)

The update_name option is set to False ! Creation of new extent not possible !
Update all subtemplates - Number of subtemplates : 26
Backup subtemplates...
Convert extent subtemplates to geojson : 100.0%Done
Backup subtemplates...Done
Update all subtemplates : 100.0%Done
Subtemplates successfully created : 0
Subtemplates successfully updated : 24
Subtemplates unsuccessfully created : 2
Subtemplates unsuccessfully updated : 0


In [ ]:
ref_geojson = "Update2025/historisierte-administrative_grenzen_k4_2025-04-06_gemeinde_4326.geojson"
number = "GDENR"
name = "GDENAME"
type = "g"
env = "dev"

import ManageAdminBoundaries as ab
import pandas as pd
import os

manage_delete = ab.UpdateSubtemplatesExtent(
    ref_geojson=ref_geojson,
    number=number,
    name=name,
    type=type,
    update_name=True,
    env=env
)

uuids = []
df = pd.read_csv(os.path.join("old_municipalities.csv"))
for index, row in df.iterrows():
    if row["MD_Linked"] == 0:
        uuids.append(f'geocatch-subtpl-extent-hoheitsgebiet-{row["GMDNR"]}')

if uuids:
    manage_delete.delete_subtemplates(uuids=uuids, with_backup=True)

In [ ]:
ref_geojson = "Update2025/historisierte-administrative_grenzen_k4_2025-04-06_bezirk_4326.geojson"
number = "GDENR"
name = "GDENAME"
type = "g"
env = "dev"

import ManageAdminBoundaries as ab
import pandas as pd
import os

manage_delete = ab.UpdateSubtemplatesExtent(
    ref_geojson=ref_geojson,
    number=number,
    name=name,
    type=type,
    update_name=True,
    env=env
)

uuids = []
df = pd.read_csv(os.path.join("old_municipalities.csv"))
for index, row in df.iterrows():
    if row["MD_Linked"] == 0:
        uuids.append(f'geocatch-subtpl-extent-hoheitsgebiet-{row["GMDNR"]}')

if uuids:
    manage_delete.delete_subtemplates(uuids=uuids, with_backup=True)

In [ ]:
# Cellule de diagnostic : tester geocatch-subtpl-extent-hoheitsgebiet-1
import ManageAdminBoundaries as ab

manage = ab.UpdateSubtemplatesExtent(
    ref_geojson="Update2025/historisierte-administrative_grenzen_k4_2025-04-06_gemeinde_4326.geojson",
    number="GDENR",
    name="GDENAME",
    type="g",
    update_name=True,
    env="dev"
)

# UUID cible
uuid = "geocatch-subtpl-extent-hoheitsgebiet-1"

# Trouver la feature correspondante (GDENR = 1)
feature = None
for f in manage.ref_geojson["features"]:
    if f["properties"]["GDENR"] == 1:
        feature = f
        break

if feature is None:
    print("❌ Aucune feature trouvée avec GDENR = 1 dans le GeoJSON !")
else:
    print(f"✅ Feature trouvée : {feature['properties']['GDENAME']}")
    
    name = feature["properties"]["GDENAME"]
    gml = ab.geojson_to_geocatgml(feature)
    
    # 1. Vérifier si l'extent existe
    print(f"\n🔍 Vérification existence de {uuid}...")
    api = ab.geopycat.geocat("dev")
    check = api.session.get(
        url=api.env + f"/geonetwork/srv/api/registries/entries/{uuid}",
        headers={"accept":"application/xml"}
    )
    
    if check.status_code == 404:
        print("⚠️ L'extent n'existe pas encore")
        
        # CRÉER L'EXTENT D'ABORD
        print("\n🆕 CRÉATION DE L'EXTENT:")
        response_create = manage.create_extent(uuid=uuid)
        print(f"Status: {response_create.status_code}")
        if hasattr(response_create, 'text'):
            print(f"Response: {response_create.text[:500]}")
        
        if response_create.status_code != 201:
            print("❌ Échec de la création, arrêt du test")
            import sys
            sys.exit()
    else:
        print(f"✅ L'extent existe déjà (status: {check.status_code})")
    
    # 2. Update extent
    print("\n🔄 TEST UPDATE:")
    response = manage.update_extent(uuid=uuid, name=name, gml=gml)
    print(f"Status: {response.status_code}")
    print(f"Response: {response.text[:800]}")
    
    # 3. Validate extent
    print("\n✅ TEST VALIDATION:")
    response = manage.validate_extent(uuid=uuid)
    print(f"Status: {response.status_code}")
    if hasattr(response, 'text'):
        print(f"Response: {response.text[:800]}")
    
    # 4. Set permissions
    print("\n🔐 TEST PERMISSIONS:")
    response = manage.set_extent_permissions(uuid=uuid)
    print(f"Status: {response.status_code}")
    if hasattr(response, 'text'):
        print(f"Response: {response.text[:500]}")
    
    # 5. Set ownership
    print("\n👤 TEST OWNERSHIP:")
    response = manage.set_extent_owner(uuid=uuid)
    print(f"Status: {response.status_code}")
    if hasattr(response, 'text'):
        print(f"Response: {response.text[:800]}")
    
    # 6. Vérification finale du XML
    print("\n📄 VÉRIFICATION XML FINAL:")
    final_check = api.session.get(
        url=api.env + f"/geonetwork/srv/api/registries/entries/{uuid}",
        headers={"accept":"application/xml"}
    )
    
    if final_check.status_code == 200:
        xml = final_check.text
        print(f"Status: {final_check.status_code}")
        print(f"XML (premiers 1500 car):\n{xml[:1500]}")
        
        # Vérifier la structure
        if 'xmlns:lan=' in xml[:500]:
            print("\n✅ Namespace 'lan' déclaré dans l'élément racine")
        else:
            print("\n❌ Namespace 'lan' PAS déclaré dans l'élément racine")
        
        if '<gco:CharacterString>' in xml:
            print("✅ Contient <gco:CharacterString> (texte par défaut présent)")
        else:
            print("❌ Pas de <gco:CharacterString> (texte par défaut manquant)")
        
        if name in xml:
            print(f"✅ Le nom '{name}' est présent dans le XML")
        else:
            print(f"❌ Le nom '{name}' n'est PAS dans le XML")
    else:
        print(f"❌ Impossible de récupérer le XML final (status: {final_check.status_code})")

✅ Feature trouvée : Aeugst am Albis

🔍 Vérification existence de geocatch-subtpl-extent-hoheitsgebiet-1...
⚠️ L'extent n'existe pas encore

🆕 CRÉATION DE L'EXTENT:
✅ Subtemplate created with UUID geocatch-subtpl-extent-hoheitsgebiet-1
Status: 201
Response: {"errors":[],"infos":[],"uuid":"05f9e342-907f-4729-8e34-ee457deb4e6e","metadata":[],"metadataErrors":{},"metadataInfos":{"48619932":[{"message":"Metadata imported from XML with UUID 'geocatch-subtpl-extent-hoheitsgebiet-1'","uuid":"geocatch-subtpl-extent-hoheitsgebiet-1","draft":true,"approved":false,"date":"2025-12-10T08:23:30.744517Z"}]},"numberOfNullRecords":0,"numberOfRecordsProcessed":1,"numberOfRecordsUnchanged":0,"numberOfRecordsWithErrors":0,"numberOfRecords":0,"numberOfRecordNotFound":0

🔄 TEST UPDATE:
Status: 201
Response: {"errors":[],"infos":[],"uuid":"e36f2b96-c309-4e10-9ed7-e556d8528b0a","metadata":[],"metadataErrors":{},"metadataInfos":{"48619932":[{"message":"Metadata imported from XML with UUID 'geocatch-subtpl-exten

In [ ]:
# Cellule de test : mettre à jour geocatch-subtpl-extent-hoheitsgebiet-6155
import ManageAdminBoundaries as ab

manage = ab.UpdateSubtemplatesExtent(
    ref_geojson="Update2025/historisierte-administrative_grenzen_k4_2025-04-06_gemeinde_4326.geojson",
    number="GDENR",
    name="GDENAME",
    type="g",
    update_name=True,
    env="dev"
)

# UUID cible
uuid = "geocatch-subtpl-extent-hoheitsgebiet-6155"

# Trouver la feature correspondante (GDENR = 6155)
feature = None
for f in manage.ref_geojson["features"]:
    if f["properties"]["GDENR"] == 6155:
        feature = f
        break

if feature is None:
    print("❌ Aucune feature trouvée avec GDENR = 6155 dans le GeoJSON !")
else:
    print(f"✅ Feature trouvée : {feature['properties']['GDENAME']}")
    
    name = feature["properties"]["GDENAME"]
    gml = ab.geojson_to_geocatgml(feature)
    
    # 1. Récupérer l'état AVANT mise à jour
    print(f"\n📋 ÉTAT AVANT MISE À JOUR:")
    api = ab.geopycat.geocat("dev")
    before = api.session.get(
        url=api.env + f"/geonetwork/srv/api/registries/entries/{uuid}",
        headers={"accept":"application/xml"}
    )
    
    if before.status_code == 200:
        xml_before = before.text
        print(f"Status: {before.status_code}")
        
        # Extraire le nom actuel
        import xml.etree.ElementTree as ET
        root_before = ET.fromstring(xml_before)
        ns = {'gco': 'http://standards.iso.org/iso/19115/-3/gco/1.0'}
        current_name = root_before.find('.//gco:CharacterString', ns)
        if current_name is not None:
            print(f"Nom actuel: '{current_name.text}'")
        
        # Vérifier si la géométrie existe
        if '<gml:posList>' in xml_before:
            print("✅ Géométrie présente")
        else:
            print("❌ Géométrie absente")
    else:
        print(f"❌ Impossible de récupérer l'extent (status: {before.status_code})")
    
    # 2. Update extent
    print(f"\n🔄 MISE À JOUR:")
    response = manage.update_extent(uuid=uuid, name=name, gml=gml)
    print(f"Status: {response.status_code}")
    if hasattr(response, 'text'):
        print(f"Response: {response.text[:500]}")
    
    # 3. Validate extent
    print("\n✅ VALIDATION:")
    response = manage.validate_extent(uuid=uuid)
    print(f"Status: {response.status_code}")
    if hasattr(response, 'text'):
        response_json = response.json()
        if "metadataErrors" in response_json and response_json["metadataErrors"]:
            print(f"❌ Erreurs de validation: {response_json['metadataErrors']}")
        else:
            print("✅ Validation réussie")
    
    # 4. Récupérer l'état APRÈS mise à jour
    print(f"\n📋 ÉTAT APRÈS MISE À JOUR:")
    after = api.session.get(
        url=api.env + f"/geonetwork/srv/api/registries/entries/{uuid}",
        headers={"accept":"application/xml"}
    )
    
    if after.status_code == 200:
        xml_after = after.text
        print(f"Status: {after.status_code}")
        
        # Extraire le nouveau nom
        root_after = ET.fromstring(xml_after)
        new_name = root_after.find('.//gco:CharacterString', ns)
        if new_name is not None:
            print(f"Nom après update: '{new_name.text}'")
            if new_name.text == name:
                print(f"✅ Le nom est correct: '{name}'")
            else:
                print(f"❌ Le nom ne correspond pas. Attendu: '{name}', Obtenu: '{new_name.text}'")
        
        # Vérifier la géométrie
        if '<gml:posList>' in xml_after:
            print("✅ Géométrie présente après update")
            
            # Extraire un extrait des coordonnées
            import re
            coords_match = re.search(r'<gml:posList[^>]*>([\d\s.]+)</gml:posList>', xml_after)
            if coords_match:
                coords = coords_match.group(1).strip().split()[:6]  # Premiers 3 points
                print(f"Premières coordonnées: {' '.join(coords)}...")
        else:
            print("❌ Géométrie absente après update")
        
        # Vérifier les namespaces
        if 'xmlns:lan=' in xml_after[:500]:
            print("✅ Namespace 'lan' déclaré")
        
        if '<gco:CharacterString>' in xml_after:
            print("✅ Structure ISO 19115-3 correcte")
    else:
        print(f"❌ Impossible de récupérer l'extent après update (status: {after.status_code})")

✅ Feature trouvée : Saint-Gingolph

📋 ÉTAT AVANT MISE À JOUR:
Status: 200
Nom actuel: 'Saint-Gingolph'
✅ Géométrie présente

🔄 MISE À JOUR:
Status: 201
Response: {"errors":[],"infos":[],"uuid":"f89e2d55-a746-4c05-a0f0-e2d38e52043b","metadata":[],"metadataErrors":{},"metadataInfos":{"34785595":[{"message":"Metadata imported from XML with UUID 'geocatch-subtpl-extent-hoheitsgebiet-6155'","uuid":"geocatch-subtpl-extent-hoheitsgebiet-6155","draft":true,"approved":false,"date":"2025-12-10T08:27:06.569346Z"}]},"numberOfNullRecords":0,"numberOfRecordsProcessed":1,"numberOfRecordsUnchanged":0,"numberOfRecordsWithErrors":0,"numberOfRecords":0,"numberOfRecordNotFo

✅ VALIDATION:
Status: 201
✅ Validation réussie

📋 ÉTAT APRÈS MISE À JOUR:
Status: 200
Nom après update: 'Saint-Gingolph'
✅ Le nom est correct: 'Saint-Gingolph'
✅ Géométrie présente après update
Premières coordonnées: 6.8053944383 46.3942966632 6.801465698 46.390064513 6.8038320074 46.3767368607...
✅ Namespace 'lan' déclaré
✅ Structure

In [2]:
# Cellule de nettoyage
import ManageAdminBoundaries as ab

manage = ab.UpdateSubtemplatesExtent(
    ref_geojson="Update2025/historisierte-administrative_grenzen_k4_2025-04-06_gemeinde_4326.geojson",
    number="GDENR",
    name="GDENAME",
    type="g",
    update_name=True,
    env="dev"
)

# Supprimer l'UUID erroné
manage.delete_subtemplates(uuids=["geocatch-subtpl-extent-commune-6266"], with_backup=False)

Delete subtemplates - Number of subtemplates : 1
Delete subtemplates : 100.0%Done
Subtemplates successfully deleted : 0
Subtemplates unsuccessfully deleted : 1
